# 006 LangChain In FastAPI

这是 LangChain 学习线的第六份 Notebook。

官方参考：

- https://docs.langchain.com/oss/python/langchain/agents
- https://docs.langchain.com/oss/python/langchain/streaming
- https://docs.langchain.com/oss/python/langchain/tools

学习目标：

1. 理解 LangChain agent 接入 FastAPI 的分层方式
2. 区分同步 invoke、异步 ainvoke 和流式 stream
3. 设计最小 Service 封装
4. 讨论如何保留 Harness 的权限审批边界
5. 判断什么时候该复用 LangChain，什么时候保留自研 Harness

---

## 1. 接入原则

不要在 FastAPI router 里直接堆 LangChain 细节。

推荐分层：

```text
Router
  -> Service
    -> Agent factory / Agent runtime
      -> Model
      -> Tools
      -> Optional governance layer
```

这和本项目现有结构一致：

```text
app/api/routers/chat_agent.py
  -> app/services/chat_agent.py
    -> app/agents/harness.py
```

In [ ]:
%pip install -U langchain langchain-openai python-dotenv

## 2. 一个最小 Agent Factory

先写一个工厂函数，负责读取 `.env` 并创建 agent。

注意：这里是教学示例，不直接改项目生产代码。

In [1]:
import os
from pathlib import Path

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI


def load_project_env() -> Path | None:
    current = Path.cwd().resolve()
    for path in [current, *current.parents]:
        env_path = path / ".env"
        if env_path.exists():
            load_dotenv(env_path, override=False)
            return env_path
    return None


@tool
def explain_fastapi_study(topic: str) -> str:
    """Explain a known concept from the fastapi-study learning project."""
    mapping = {
        "router": "Router 负责 HTTP 入口和参数声明，不应堆业务细节。",
        "service": "Service 负责业务编排，适合封装 agent 调用。",
        "harness": "Harness 负责 agent 控制流、工具权限、ledger 和恢复。",
        "tool": "Tool 是受管执行接口，模型只提出调用请求。",
    }
    return mapping.get(topic.strip().lower(), f"暂时没有 {topic!r} 的固定说明。")


def create_study_agent():
    load_project_env()
    api_key = os.getenv("OPENAI_API_KEY")
    model_name = os.getenv("OPENAI_MODEL", "gpt-5.4-mini")
    base_url = os.getenv("OPENAI_BASE_URL") or None
    if not api_key:
        return None
    model = ChatOpenAI(model=model_name, api_key=api_key, base_url=base_url)
    return create_agent(
        model=model,
        tools=[explain_fastapi_study],
        system_prompt="你是一个中文教学助手，回答要简洁。需要项目概念时可以使用工具。",
    )


agent = create_study_agent()
print("agent ready =", agent is not None)

agent ready = True


## 3. Service 封装

FastAPI 的 router 不应该直接知道 LangChain 的调用细节。

更好的方式是封装一个 Service。

In [2]:
class LangChainStudyService:
    def __init__(self, agent):
        self.agent = agent

    def chat(self, message: str) -> dict:
        if self.agent is None:
            return {
                "answer": "LangChain agent 未初始化，请先配置 OPENAI_API_KEY。",
                "raw_messages": [],
            }
        result = self.agent.invoke({"messages": [{"role": "user", "content": message}]})
        messages = result.get("messages", [])
        answer = messages[-1].content if messages else ""
        return {
            "answer": answer,
            "raw_messages": [str(item) for item in messages],
        }


service = LangChainStudyService(agent)
print(service.chat("请解释 service"))

{'answer': '\n\nService 负责业务编排，适合封装 agent 调用。', 'raw_messages': ["content='请解释 service' additional_kwargs={} response_metadata={} id='374dbecc-cbb7-4294-83b1-e8e4cb9ea231'", "content='\\n\\n' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 567, 'prompt_tokens': 301, 'total_tokens': 868, 'completion_tokens_details': None, 'prompt_tokens_details': None, 'reasoning_tokens': 0}, 'model_provider': 'openai', 'model_name': 'qwq', 'system_fingerprint': None, 'id': 'chatcmpl-ChRF37RDbVhiQqDTxRM3am', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--019e6775-b3fa-7fc3-acd1-37a86615f1d1-0' tool_calls=[{'name': 'explain_fastapi_study', 'args': {'topic': 'service'}, 'id': 'call_124307a3483e481cbf2f6aad', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 301, 'output_tokens': 567, 'total_tokens': 868, 'input_token_details': {}, 'output_token_details': {}}", "content='Service 负责业务编排，适合封装 agent 调用。' name='explain_fastapi_

## 4. Router 应该长什么样

下面是 FastAPI router 的形状示例。

这段代码只是教学展示，不会自动注册到当前项目。

In [3]:
from pydantic import BaseModel, Field


class LangChainChatRequest(BaseModel):
    message: str = Field(..., min_length=1)


class LangChainChatResponse(BaseModel):
    answer: str


print(LangChainChatRequest(message="hello"))
print(LangChainChatResponse(answer="world"))

message='hello'
answer='world'


Router 伪代码：

```python
router = APIRouter(prefix="/langchain-study", tags=["LangChain 教学"])

@router.post("/chat", response_model=LangChainChatResponse)
def chat(payload: LangChainChatRequest):
    result = langchain_study_service.chat(payload.message)
    return LangChainChatResponse(answer=result["answer"])
```

如果底层使用同步 `agent.invoke(...)`，router 用普通 `def` 就够了，FastAPI 会把同步 endpoint 放到线程池。

## 5. 同步、异步和流式

LangChain 常见调用方式：

| 方式 | 场景 | FastAPI 建议 |
|---|---|---|
| `invoke` | 普通同步调用 | `def` endpoint |
| `ainvoke` | 全链路 async | `async def` endpoint |
| `stream` | 同步流式事件 | `StreamingResponse` |
| `astream` | 异步流式事件 | `async StreamingResponse` |

不要只为了看起来高级就把所有接口改成 `async def`。

如果底层调用是同步阻塞的，普通 `def` 反而更清晰。

## 6. 流式返回应该怎么想

本仓库现有 `/api/v1/chat-agent/chat/stream` 返回的是 Harness SSE 事件：

```text
ledger
model_request
model_response
plan
permission
tool_result
answer_delta
done
```

LangChain stream 通常返回 agent 内部更新或 token/event。

如果要接入当前页面，建议做一层事件适配：

```text
LangChain stream event
  -> convert to Harness-style SSE event
  -> frontend remains stable
```

## 7. 关键问题：权限审批放在哪里

LangChain 可以帮你做 tool calling，但本仓库的审批链路不能丢：

```text
tool_call
  -> ToolRegistry.decide_permission()
  -> allow / ask / deny
  -> ApprovalTicket
  -> resume_approval
```

所以 LangChain 接入当前项目时，不建议让高风险工具直接由 LangChain 裸执行。

更稳妥的设计是：

```text
LangChain 负责生成工具调用意图
Harness 负责权限裁决和执行恢复
```

## 8. 最小落地路线

如果后面真的把 LangChain 接进项目，推荐按这个顺序：

1. 新增教学 endpoint，不替换现有 Harness agent。
2. 只接低风险只读 tool。
3. 先做非流式 `/langchain-study/chat`。
4. 再做 `/langchain-study/chat/stream`。
5. 最后再讨论如何复用现有 `ToolRegistry` 和 approval。

这样不会把当前已经能运行的 Harness 智能体搞复杂。

## 9. 本讲小结

这一讲记住四点：

1. LangChain 接入 FastAPI 时，router 不应该直接堆 agent 细节。
2. `invoke / ainvoke / stream / astream` 要和 FastAPI endpoint 形态匹配。
3. LangChain tool calling 不等于业务级审批治理。
4. 当前项目更适合先新增 LangChain 教学 endpoint，而不是替换 Harness runtime。

到这里，LangChain 第一阶段学习线已经形成：

```text
001 overview
002 models/messages
003 tools
004 structured output
005 agents/control flow
006 FastAPI integration
```